
# HealthDoc AI — Open-Source RAG Document Q&A System

This notebook builds a complete **Document Q&A chatbot** for healthcare / pharmaceutical PDFs.

It includes:

- **Digital PDF extraction**
- **OCR fallback for scanned PDF pages**
- **Document-type metadata tagging**
- **Logical document grouping**
- **Overlapping text chunks**
- **Open-source SentenceTransformer embeddings**
- **FAISS semantic retrieval**
- **Document-type filtering**
- **Automatic query routing**
- **Configurable number of retrieved chunks**
- **Open-source FLAN-T5 language model — no Gemini**
- **Source citations**
- **Confidence score**
- **Chunk count**
- **Visible chat history**
- **Chat-history export**
- **Professional Gradio interface**

> **Important for Google Colab:** the final launch cell uses `inline=False` intentionally. Open the generated **gradio.live** link in a new browser tab. This avoids a common Colab iframe upload problem that can appear as **“Failed to fetch”**.


In [ ]:

# ============================================
# STEP 1: Install a stable Colab environment
# ============================================
# IMPORTANT:
# We pin Gradio 5.49.1 so the UI/API does not change underneath this notebook.
# This avoids the Gradio 6 component changes that caused the earlier UI issues.

!apt-get -qq update
!apt-get -qq install -y tesseract-ocr

!pip install -q --upgrade \
    "gradio==5.49.1" \
    "PyMuPDF>=1.26" \
    "pytesseract>=0.3.13" \
    "Pillow>=11" \
    "sentence-transformers>=5,<6" \
    "faiss-cpu>=1.11" \
    "transformers>=4.55,<5" \
    "sentencepiece>=0.2" \
    "accelerate>=1.0" \
    "langchain-text-splitters>=0.3"

print("Stable environment installed.")
print("IMPORTANT: If this is the first run, use Runtime > Restart session,")
print("then continue from STEP 2. You only need to restart once.")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.4/325.4 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

In [ ]:

# ============================================
# STEP 2: Imports + Data Structures
# ============================================
#
# WHAT THIS DOES:
# Imports all libraries and defines the metadata objects used
# throughout the RAG pipeline.
# ============================================

import os
import re
import json
import time
from pathlib import Path
from dataclasses import dataclass
from datetime import datetime
from typing import List, Dict, Any, Optional, Tuple

import numpy as np
import fitz
import pytesseract
from PIL import Image
import faiss
import torch
import gradio as gr

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_text_splitters import RecursiveCharacterTextSplitter


@dataclass
class ChunkMetadata:
    text: str
    doc_type: str
    source_file: str
    source_id: str
    doc_id: str
    page_start: int
    page_end: int
    chunk_index: int
    scanned_ocr: bool = False


print("Libraries loaded successfully.")
print("Gradio version:", gr.__version__)


assert gr.__version__.startswith("5."), f"Expected Gradio 5.x, got {gr.__version__}"


Libraries loaded successfully.
Gradio version: 5.49.1


In [ ]:
# ============================================
# STEP 3: Load Open-Source Models
# ============================================
#
# MODEL 1 — EMBEDDINGS
# sentence-transformers/all-MiniLM-L6-v2
#
# Used for:
# - converting document chunks into vectors
# - semantic search with FAISS
#
#
# MODEL 2 — GENERATIVE LLM
# google/flan-t5-small
#
# Used mainly for:
# - summary mode
# - optional natural-language generation
#
#
# MODEL 3 — EXTRACTIVE QUESTION ANSWERING
# deepset/roberta-base-squad2
#
# Used for:
# - finding the EXACT answer inside retrieved text
#
# Example:
#
# Question:
# "How much did we earn in Q3 2024?"
#
# Context:
# "...Q3 2024 net revenue was $31.44 million..."
#
# Extracted answer:
# "$31.44 million"
#
# This prevents FLAN-T5 from copying random text
# such as "Black Friday..." or "EVIDENCE 3..."
#
# NO GEMINI IS USED.
# ============================================


from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    pipeline
)

import torch


# ============================================
# MODEL NAMES
# ============================================

EMBEDDING_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

LLM_MODEL_NAME = (
    "google/flan-t5-base"
)

QA_MODEL_NAME = (
    "deepset/roberta-base-squad2"
)


# ============================================
# DEVICE
# ============================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Using device:", device)


# ============================================
# 1. EMBEDDING MODEL
# ============================================

print("\nLoading embedding model...")

embedder = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print(
    "Embedding model ready:",
    EMBEDDING_MODEL_NAME
)


# ============================================
# 2. FLAN-T5 GENERATIVE MODEL
# ============================================

print("\nLoading FLAN-T5...")

tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL_NAME
)

llm_model = (
    AutoModelForSeq2SeqLM
    .from_pretrained(
        LLM_MODEL_NAME
    )
)

llm_model = llm_model.to(
    device
)

llm_model.eval()


def generate_with_llm(
    prompt,
    max_new_tokens=220
):
    """
    Generate text using FLAN-T5.

    We keep this function because it can still
    be useful for Summary Mode.
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {
        key: value.to(device)
        for key, value
        in inputs.items()
    }

    with torch.no_grad():

        output_ids = llm_model.generate(

            **inputs,

            max_new_tokens=max_new_tokens,

            do_sample=False,

            num_beams=2,

            repetition_penalty=1.05
        )


    answer = tokenizer.decode(

        output_ids[0],

        skip_special_tokens=True
    )


    return answer.strip()


print(
    "Generative model ready:",
    LLM_MODEL_NAME
)


# ============================================
# 3. EXTRACTIVE QA MODEL
# ============================================

print("\nLoading question-answering model...")


# Hugging Face pipeline uses:
#   device = 0  -> GPU
#   device = -1 -> CPU

qa_device = (
    0
    if torch.cuda.is_available()
    else -1
)


qa_model = pipeline(

    task="question-answering",

    model=QA_MODEL_NAME,

    tokenizer=QA_MODEL_NAME,

    device=qa_device
)


print(
    "QA model ready:",
    QA_MODEL_NAME
)


# ============================================
# HELPER FUNCTION
# ============================================

def extract_answer(
    question,
    context
):
    """
    Find the exact answer inside one document chunk.

    Returns:
        answer
        confidence score
    """

    if not context or not context.strip():

        return {
            "answer": "",
            "score": 0.0
        }


    try:

        result = qa_model(

            question=question,

            context=context
        )


        return {

            "answer":
                result.get(
                    "answer",
                    ""
                ).strip(),

            "score":
                float(
                    result.get(
                        "score",
                        0.0
                    )
                )
        }


    except Exception as e:

        print(
            "QA extraction error:",
            e
        )

        return {
            "answer": "",
            "score": 0.0
        }


# ============================================
# TEST MODEL
# ============================================

test_result = extract_answer(

    question=
        "How much was the revenue?",

    context=
        "The company reported revenue of "
        "$31.44 million during the quarter."
)


print("\n------------------------------")
print("MODEL TEST")
print("------------------------------")

print(
    "Extracted answer:",
    test_result["answer"]
)

print(
    "QA confidence:",
    round(
        test_result["score"] * 100,
        1
    ),
    "%"
)


print("\n==============================")
print("ALL MODELS READY")
print("==============================")

print(
    "Embedding:",
    EMBEDDING_MODEL_NAME
)

print(
    "Generative LLM:",
    LLM_MODEL_NAME
)

print(
    "Extractive QA:",
    QA_MODEL_NAME
)

print(
    "Device:",
    device
)

Using device: cpu

Loading embedding model...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Device set to use cpu


QA model ready: deepset/roberta-base-squad2

------------------------------
MODEL TEST
------------------------------
Extracted answer: $31.44 million
QA confidence: 80.3 %

ALL MODELS READY
Embedding: sentence-transformers/all-MiniLM-L6-v2
Generative LLM: google/flan-t5-base
Extractive QA: deepset/roberta-base-squad2
Device: cpu


In [ ]:

# ============================================
# STEP 4: Document-Type Metadata Tagging
# ============================================
#
# WHAT THIS DOES:
# Assigns a document type to each extracted PDF page.
#
# This uses deterministic keyword rules instead of Gemini.
# The metadata is later used for:
# - document grouping
# - filtering
# - query routing
# - source display
# ============================================

DOCUMENT_TYPES = [
    "Cover Letter",
    "Certificate of Quality",
    "Packaging Specification",
    "BSE/TSE Declaration",
    "Material Description",
    "Supplier Qualification",
    "Chain of Custody",
    "Other"
]

DOC_TYPE_KEYWORDS = {
    "Cover Letter": [
        "to whom it may concern",
        "dear ",
        "re:",
        "sincerely",
        "letter"
    ],
    "Certificate of Quality": [
        "certificate of quality",
        "certificate of analysis",
        "quality certificate",
        "lot number",
        "batch number",
        "meets specification",
        "conforms"
    ],
    "Packaging Specification": [
        "packaging specification",
        "packaging component",
        "packaging configuration",
        "blister tray",
        "lid film",
        "secondary carton",
        "drawing change",
        "configuration change"
    ],
    "BSE/TSE Declaration": [
        "bse",
        "tse",
        "transmissible spongiform",
        "bovine spongiform",
        "animal derived",
        "animal-derived"
    ],
    "Material Description": [
        "material description",
        "material specification",
        "material name",
        "chemical composition",
        "material properties"
    ],
    "Supplier Qualification": [
        "supplier qualification",
        "supplier assessment",
        "approved supplier",
        "supplier audit",
        "vendor qualification"
    ],
    "Chain of Custody": [
        "chain of custody",
        "custody transfer",
        "traceability",
        "received by",
        "released by"
    ]
}


def classify_doc_type(text: str) -> str:
    lowered = re.sub(r"\s+", " ", text.lower())
    scores = {}

    for doc_type, keywords in DOC_TYPE_KEYWORDS.items():
        score = 0

        for keyword in keywords:
            if keyword in lowered:
                score += 2 if " " in keyword.strip() else 1

        scores[doc_type] = score

    best_type = max(scores, key=scores.get)

    if scores[best_type] == 0:
        return "Other"

    return best_type


print("Document metadata classifier ready.")


Document metadata classifier ready.


In [ ]:

# ============================================
# STEP 5: PDF Processing + OCR Fallback
# ============================================
#
# WHAT THIS DOES:
# 1. Opens every uploaded PDF.
# 2. Extracts normal digital text with PyMuPDF.
# 3. If a page contains very little text, renders the page and
#    applies Tesseract OCR.
# 4. Tags every page with document type and source metadata.
#
# This supports BOTH digital PDFs and scanned/image PDFs.
# ============================================

MIN_DIGITAL_TEXT_CHARS = 40


def clean_text(text: str) -> str:
    if not text:
        return ""

    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def ocr_pdf_page(page: fitz.Page, dpi: int = 180) -> str:
    pix = page.get_pixmap(dpi=dpi, alpha=False)

    image = Image.frombytes(
        "RGB",
        [pix.width, pix.height],
        pix.samples
    )

    return clean_text(
        pytesseract.image_to_string(image)
    )


def process_pdf_file(file_path: str) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    file_path = str(file_path)
    source_file = os.path.basename(file_path)

    pages = []
    ocr_pages = 0

    pdf = fitz.open(file_path)

    try:
        for page_index in range(len(pdf)):
            page = pdf[page_index]

            text = clean_text(page.get_text("text"))
            used_ocr = False

            if len(text) < MIN_DIGITAL_TEXT_CHARS:
                ocr_text = ocr_pdf_page(page)

                if len(ocr_text) > len(text):
                    text = ocr_text
                    used_ocr = True
                    ocr_pages += 1

            doc_type = classify_doc_type(text)

            pages.append({
                "source_file": source_file,
                "source_id": f"{source_file}::page_{page_index + 1}",
                "page": page_index + 1,
                "text": text,
                "doc_type": doc_type,
                "scanned_ocr": used_ocr
            })

    finally:
        pdf.close()

    file_info = {
        "source_file": source_file,
        "page_count": len(pages),
        "ocr_pages": ocr_pages
    }

    return pages, file_info


print("PDF extraction and OCR functions ready.")


PDF extraction and OCR functions ready.


In [ ]:

# ============================================
# STEP 6: Logical Documents + Metadata Chunks
# ============================================
#
# WHAT THIS DOES:
# 1. Groups consecutive pages that appear to be the same document.
# 2. Assigns each logical document a doc_id.
# 3. Splits every page into overlapping chunks.
# 4. Attaches source, doc type, doc ID, page range, chunk index,
#    and OCR metadata.
#
# Page-level chunking keeps source citations precise.
# ============================================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=650,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""]
)


def group_logical_documents(pages: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    logical_docs = []
    current = None
    counter = 0

    for page in pages:
        starts_new = (
            current is None
            or current["source_file"] != page["source_file"]
            or current["doc_type"] != page["doc_type"]
        )

        if starts_new:
            current = {
                "doc_id": f"doc_{counter:03d}",
                "source_file": page["source_file"],
                "doc_type": page["doc_type"],
                "page_start": page["page"],
                "page_end": page["page"],
                "pages": [page["page"]]
            }

            logical_docs.append(current)
            counter += 1

        else:
            current["page_end"] = page["page"]
            current["pages"].append(page["page"])

    return logical_docs


def attach_doc_ids_to_pages(
    pages: List[Dict[str, Any]],
    logical_docs: List[Dict[str, Any]]
) -> List[Dict[str, Any]]:
    lookup = {}

    for doc in logical_docs:
        for page_num in doc["pages"]:
            lookup[(doc["source_file"], page_num)] = doc["doc_id"]

    for page in pages:
        page["doc_id"] = lookup[
            (page["source_file"], page["page"])
        ]

    return pages


def chunk_pages_with_metadata(
    pages: List[Dict[str, Any]]
) -> List[ChunkMetadata]:
    chunks = []
    chunk_index = 0

    for page in pages:
        if not page["text"].strip():
            continue

        page_chunks = splitter.split_text(page["text"])

        for chunk_text in page_chunks:
            chunks.append(
                ChunkMetadata(
                    text=chunk_text,
                    doc_type=page["doc_type"],
                    source_file=page["source_file"],
                    source_id=page["source_id"],
                    doc_id=page["doc_id"],
                    page_start=page["page"],
                    page_end=page["page"],
                    chunk_index=chunk_index,
                    scanned_ocr=page["scanned_ocr"]
                )
            )

            chunk_index += 1

    return chunks


print("Logical document grouping and chunking ready.")


Logical document grouping and chunking ready.


In [ ]:

# ============================================
# STEP 7: Embeddings + FAISS
# ============================================
#
# WHAT THIS DOES:
# Converts all text chunks to embedding vectors and stores them
# in a FAISS index for semantic retrieval.
# ============================================

def normalize_embeddings(vectors: np.ndarray) -> np.ndarray:
    vectors = np.asarray(vectors, dtype="float32")

    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1.0

    return vectors / norms


def build_faiss_index(
    chunks: List[ChunkMetadata]
) -> faiss.IndexFlatIP:
    texts = [chunk.text for chunk in chunks]

    vectors = embedder.encode(
        texts,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    vectors = normalize_embeddings(vectors)

    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)

    return index


print("FAISS indexing functions ready.")


FAISS indexing functions ready.


In [ ]:

# ============================================
# STEP 8: Query Routing + Retrieval
# ============================================
#
# SETTINGS SUPPORTED:
# - Manual document-type filter
# - Auto-route query
# - Number of chunks to retrieve
#
# Retrieval returns ChunkMetadata + similarity score.
# ============================================

ROUTING_KEYWORDS = {
    "Packaging Specification": [
        "packaging", "package", "blister", "carton", "label",
        "configuration", "tray", "lid film"
    ],
    "Certificate of Quality": [
        "quality", "certificate", "lot", "batch", "specification",
        "test result", "conformance"
    ],
    "BSE/TSE Declaration": [
        "bse", "tse", "animal derived", "bovine", "spongiform"
    ],
    "Material Description": [
        "material", "composition", "material description", "properties"
    ],
    "Supplier Qualification": [
        "supplier", "vendor", "qualification", "audit"
    ],
    "Chain of Custody": [
        "chain of custody", "custody", "traceability", "transfer"
    ],
    "Cover Letter": [
        "letter", "correspondence", "who wrote", "recipient"
    ]
}


def route_query(query: str) -> Optional[str]:
    q = query.lower()
    scores = {}

    for doc_type, keywords in ROUTING_KEYWORDS.items():
        scores[doc_type] = sum(
            1 for keyword in keywords
            if keyword in q
        )

    best_type = max(scores, key=scores.get)

    if scores[best_type] == 0:
        return None

    return best_type


def retrieve_chunks(
    query: str,
    chunks: List[ChunkMetadata],
    index: faiss.IndexFlatIP,
    k: int = 4,
    filter_doc_type: Optional[str] = None,
    auto_route: bool = True
) -> Tuple[List[Tuple[ChunkMetadata, float]], Optional[str]]:

    if not chunks or index is None:
        return [], None

    active_type = filter_doc_type

    if active_type is None and auto_route:
        active_type = route_query(query)

    query_vector = embedder.encode(
        [query],
        convert_to_numpy=True,
        show_progress_bar=False
    )

    query_vector = normalize_embeddings(query_vector)

    candidate_k = min(
        len(chunks),
        max(int(k) * 6, int(k))
    )

    scores, indices = index.search(
        query_vector,
        candidate_k
    )

    retrieved = []

    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue

        chunk = chunks[int(idx)]

        if active_type and chunk.doc_type != active_type:
            continue

        retrieved.append(
            (chunk, float(score))
        )

        if len(retrieved) >= int(k):
            break

    # If auto-routing was too restrictive, retry across all document types.
    if not retrieved and filter_doc_type is None and active_type is not None:
        for score, idx in zip(scores[0], indices[0]):
            if idx < 0:
                continue

            retrieved.append(
                (chunks[int(idx)], float(score))
            )

            if len(retrieved) >= int(k):
                break

        active_type = None

    return retrieved, active_type


print("Routing and retrieval functions ready.")


Routing and retrieval functions ready.


In [ ]:
# ============================================
# STEP 9: RAG Answer Generation
# ============================================
#
# BEHAVIOR:
#
# Filter = All
# -> retrieve from all documents
# -> DO NOT refuse just because similarity is low
# -> try to answer from the best retrieved evidence
#
# Filter = specific document type
# -> if retrieval is weak, refuse like the sample chatbot
#
# Then:
# 1. RoBERTa extracts the exact answer
# 2. FLAN-T5 rewrites it naturally
# 3. Sources + confidence are returned
# ============================================


MIN_RELEVANCE = 0.45


# ============================================
# FIND BEST EXTRACTED ANSWER
# ============================================

def find_best_extracted_answer(
    question,
    retrieved
):

    results = []


    for chunk, retrieval_score in retrieved:

        try:

            qa_result = extract_answer(
                question,
                chunk.text
            )


            answer = qa_result[
                "answer"
            ].strip()


            qa_score = float(
                qa_result[
                    "score"
                ]
            )


            # Skip empty/bad answers
            if not answer:
                continue


            if answer in [
                ".",
                ",",
                "-",
                ":"
            ]:
                continue


            # --------------------------------
            # Numeric-answer bonus
            # --------------------------------

            numeric_bonus = 0.0


            if (
                "how much" in question.lower()
                and re.search(
                    r"\d",
                    answer
                )
            ):

                numeric_bonus = 0.20


            # --------------------------------
            # Combined answer-selection score
            # --------------------------------

            combined_score = (

                0.70 * qa_score

                +

                0.30 * max(
                    float(
                        retrieval_score
                    ),
                    0.0
                )

                +

                numeric_bonus
            )


            results.append({

                "answer":
                    answer,

                "qa_score":
                    qa_score,

                "retrieval_score":
                    float(
                        retrieval_score
                    ),

                "combined_score":
                    combined_score,

                "chunk":
                    chunk
            })


        except Exception as e:

            print(
                "QA extraction error:",
                e
            )


    if not results:
        return None


    results.sort(

        key=lambda x:
            x["combined_score"],

        reverse=True
    )


    return results[0]



# ============================================
# BUILD SOURCES
# ============================================

def build_sources(
    retrieved
):

    sources = []


    for i, (
        chunk,
        score
    ) in enumerate(
        retrieved,
        start=1
    ):


        if (
            chunk.page_start
            == chunk.page_end
        ):

            pages = str(
                chunk.page_start
            )

        else:

            pages = (
                f"{chunk.page_start}"
                f"-{chunk.page_end}"
            )


        sources.append({

            "source_id":
                f"Source {i}",

            "source_file":
                chunk.source_file,

            "doc_type":
                chunk.doc_type,

            "pages":
                pages,

            "similarity":
                round(
                    float(score),
                    3
                ),

            "ocr":
                chunk.scanned_ocr
        })


    return sources



# ============================================
# NATURAL-LANGUAGE ANSWER
# ============================================

def generate_natural_answer(
    question,
    extracted_answer,
    best_chunk
):

    prompt = f"""
You are answering a question about a document.

Use ONLY the information below.

QUESTION:
{question}

EXACT ANSWER FOUND:
{extracted_answer}

RELEVANT DOCUMENT TEXT:
{best_chunk.text}

Write a clear and complete answer.

Rules:
- Answer the question directly.
- You MUST include this exact answer:
  {extracted_answer}
- Include useful context from the document if available,
  such as company name, quarter, year, revenue type,
  product name, or document description.
- Do not invent information.
- Do not mention retrieval.
- Do not mention chunks.
- Do not mention evidence.
- Do not mention source numbers.
- Do not repeat the question.
- Write only 1 or 2 natural sentences.

ANSWER:
""".strip()


    natural_answer = generate_with_llm(

        prompt,

        max_new_tokens=100
    )


    return natural_answer.strip()



# ============================================
# MAIN RAG ANSWER FUNCTION
# ============================================

def generate_rag_answer(
    question,
    retrieved,
    mode="Q&A Mode",
    filter_is_active=False
):


    # ========================================
    # 1. NOTHING RETRIEVED
    # ========================================

    if not retrieved:

        return {

            "answer": (
                "Sorry, I could not find relevant "
                "information in the documents "
                "to answer this question."
            ),

            "sources": [],

            "confidence": 0.0,

            "chunks_used": 0
        }


    # ========================================
    # 2. BUILD SOURCES
    # ========================================

    sources = build_sources(
        retrieved
    )


    # ========================================
    # 3. RETRIEVAL SCORES
    # ========================================

    retrieval_scores = [

        max(
            float(score),
            0.0
        )

        for _, score
        in retrieved
    ]


    best_retrieval_score = max(
        retrieval_scores
    )


    # ========================================
    # 4. REFUSE ONLY WHEN A SPECIFIC FILTER
    #    IS ACTIVE
    #
    # IMPORTANT:
    # Filter = All
    # -> filter_is_active = False
    # -> DO NOT refuse here
    #
    # Filter = specific type
    # -> filter_is_active = True
    # -> refuse if relevance is weak
    # ========================================

    if (
        filter_is_active
        and best_retrieval_score < MIN_RELEVANCE
    ):

        return {

            "answer": (
                "Sorry, but the selected document type "
                "does not contain enough relevant information "
                "to answer this question."
            ),

            "sources":
                sources,

            "confidence":
                float(
                    np.clip(
                        best_retrieval_score,
                        0.0,
                        1.0
                    )
                ),

            "chunks_used":
                len(retrieved)
        }


    # ========================================
    # 5. SUMMARY MODE
    # ========================================

    if mode == "Summary Mode":


        context = "\n\n".join(

            chunk.text

            for chunk, _
            in retrieved[:3]
        )


        prompt = f"""
Summarize the document information below.

Use ONLY the provided document text.

DOCUMENT TEXT:
{context}

USER REQUEST:
{question}

Rules:
- Give a short and clear summary.
- Do not invent information.
- Do not mention retrieval.
- Do not mention chunks.
- Do not mention source numbers.

SUMMARY:
""".strip()


        answer = generate_with_llm(

            prompt,

            max_new_tokens=140
        )


        confidence = float(

            np.clip(

                np.mean(
                    retrieval_scores
                ),

                0.0,

                1.0
            )
        )


        return {

            "answer":
                answer.strip(),

            "sources":
                sources,

            "confidence":
                confidence,

            "chunks_used":
                len(retrieved)
        }


    # ========================================
    # 6. Q&A MODE
    # ========================================

    best_result = find_best_extracted_answer(

        question,

        retrieved
    )


    # ========================================
    # 7. NO USABLE QA ANSWER
    # ========================================

    if best_result is None:

        return {

            "answer": (
                "Sorry, but I could not find enough "
                "information in the documents "
                "to answer this question."
            ),

            "sources":
                sources,

            "confidence":
                float(
                    np.clip(
                        best_retrieval_score,
                        0.0,
                        1.0
                    )
                ),

            "chunks_used":
                len(retrieved)
        }


    # ========================================
    # 8. GET EXACT ANSWER
    # ========================================

    extracted_answer = (
        best_result[
            "answer"
        ]
    )


    best_chunk = (
        best_result[
            "chunk"
        ]
    )


    qa_confidence = float(
        best_result[
            "qa_score"
        ]
    )


    answer_retrieval_confidence = float(

        max(
            best_result[
                "retrieval_score"
            ],
            0.0
        )
    )


    # ========================================
    # 9. SECOND REFUSAL CHECK
    #
    # AGAIN:
    # only apply this when a specific
    # document filter is active
    # ========================================

    if (
        filter_is_active
        and qa_confidence < 0.01
        and answer_retrieval_confidence < 0.50
    ):

        return {

            "answer": (
                "Sorry, but the selected document type "
                "does not provide a reliable answer "
                "to this question."
            ),

            "sources":
                sources,

            "confidence":
                answer_retrieval_confidence,

            "chunks_used":
                len(retrieved)
        }


    # ========================================
    # 10. NATURAL ANSWER
    # ========================================

    answer = generate_natural_answer(

        question,

        extracted_answer,

        best_chunk
    )


    # ========================================
    # 11. FALLBACK
    #
    # Make sure FLAN-T5 keeps the
    # extracted answer.
    # ========================================

    if (
        not answer
        or extracted_answer.lower()
        not in answer.lower()
    ):

        answer = (

            f"Based on the document, "
            f"the answer is **{extracted_answer}**."
        )


    # ========================================
    # 12. CONFIDENCE
    # ========================================

    numeric_bonus = 0.0


    if (
        "how much" in question.lower()

        and re.search(
            r"\d",
            extracted_answer
        )
    ):

        numeric_bonus = 0.15


    confidence = (

        0.55
        * qa_confidence

        +

        0.45
        * answer_retrieval_confidence

        +

        numeric_bonus
    )


    confidence = float(

        np.clip(

            confidence,

            0.0,

            1.0
        )
    )


    # ========================================
    # 13. RETURN RESULT
    # ========================================

    return {

        "answer":
            answer,

        "sources":
            sources,

        "confidence":
            confidence,

        "chunks_used":
            len(retrieved)
    }


print(
    "Step 9 ready."
)

print(
    "All = answer normally."
)

print(
    "Specific weak filter = refuse."
)

Step 9 ready.
All = answer normally.
Specific weak filter = refuse.


In [ ]:

# ============================================
# STEP 10: Enhanced Document Store
# ============================================
#
# WHAT THIS DOES:
# Creates ONE central in-memory document store.
#
# IMPORTANT:
# FAISS stays inside this Python object.
# It is NOT passed through gr.State.
# ============================================

class EnhancedDocumentStore:

    def __init__(self):
        self.pages = []
        self.logical_docs = []
        self.chunks = []
        self.index = None
        self.file_infos = []
        self.processing_stats = {}
        self.is_ready = False


    def clear(self):
        self.pages = []
        self.logical_docs = []
        self.chunks = []
        self.index = None
        self.file_infos = []
        self.processing_stats = {}
        self.is_ready = False


    def process_files(self, files) -> Tuple[bool, Dict[str, Any]]:
        self.clear()
        start_time = time.time()

        try:
            if files is None:
                return False, {"error": "Please upload at least one PDF."}

            if isinstance(files, (str, Path)):
                files = [str(files)]

            files = [str(f) for f in files if f]

            if not files:
                return False, {"error": "Please upload at least one PDF."}

            # 1. Extract PDFs + OCR when needed
            for file_path in files:
                print("Processing:", os.path.basename(file_path))

                pages, file_info = process_pdf_file(file_path)

                self.pages.extend(pages)
                self.file_infos.append(file_info)

            # 2. Reconstruct logical documents
            self.logical_docs = group_logical_documents(
                self.pages
            )

            self.pages = attach_doc_ids_to_pages(
                self.pages,
                self.logical_docs
            )

            # 3. Chunk with metadata
            self.chunks = chunk_pages_with_metadata(
                self.pages
            )

            if not self.chunks:
                return False, {
                    "error": "No readable text was found in the uploaded PDF(s)."
                }

            # 4. Build FAISS index
            self.index = build_faiss_index(
                self.chunks
            )

            # 5. Processing statistics
            elapsed = time.time() - start_time

            self.processing_stats = {
                "filenames": [
                    info["source_file"]
                    for info in self.file_infos
                ],
                "total_pages": len(self.pages),
                "documents_found": len(self.logical_docs),
                "total_chunks": len(self.chunks),
                "ocr_pages": sum(
                    info["ocr_pages"]
                    for info in self.file_infos
                ),
                "document_types": sorted(
                    set(
                        doc["doc_type"]
                        for doc in self.logical_docs
                    )
                ),
                "processing_time": elapsed
            }

            self.is_ready = True

            return True, self.processing_stats

        except Exception as exc:
            self.is_ready = False
            print("PROCESSING ERROR:", repr(exc))

            return False, {
                "error": f"{type(exc).__name__}: {exc}"
            }


    def get_document_structure(self) -> List[Dict[str, Any]]:
        rows = []

        for doc in self.logical_docs:
            chunk_count = sum(
                1 for chunk in self.chunks
                if chunk.doc_id == doc["doc_id"]
            )

            rows.append({
                "doc_id": doc["doc_id"],
                "doc_type": doc["doc_type"],
                "source_file": doc["source_file"],
                "pages": (
                    str(doc["page_start"])
                    if doc["page_start"] == doc["page_end"]
                    else f'{doc["page_start"]}-{doc["page_end"]}'
                ),
                "chunks": chunk_count
            })

        return rows


    def query(
        self,
        question: str,
        document_filter: str = "All",
        auto_route: bool = True,
        k: int = 4,
        mode: str = "Q&A Mode"
    ) -> Dict[str, Any]:

        if not self.is_ready:
            return {
                "answer": "Please upload and process a PDF first.",
                "sources": [],
                "confidence": 0.0,
                "chunks_used": 0,
                "route": "None"
            }

        manual_filter = (
            None
            if not document_filter or document_filter == "All"
            else document_filter
        )

        retrieved, routed_type = retrieve_chunks(
            query=question,
            chunks=self.chunks,
            index=self.index,
            k=int(k),
            filter_doc_type=manual_filter,
            auto_route=bool(auto_route)
        )

        result = generate_rag_answer(
            question=question,
            retrieved=retrieved,
            mode=mode,

            # True only when the user manually
            # selected a specific document type.
            filter_is_active=(
                manual_filter is not None
            )
        )

        result["route"] = (
            manual_filter
            or routed_type
            or "All documents"
        )

        return result


doc_store = EnhancedDocumentStore()

print("EnhancedDocumentStore ready.")


EnhancedDocumentStore ready.


In [ ]:
# ============================================
# STEP 10A: Retrieval Evaluation Test Set
# ============================================
#
# WHAT WE'RE DOING:
# We manually create 5 questions where we KNOW
# which page contains the correct answer.
#
# This is our "answer key" for evaluating retrieval.
# ============================================

evaluation_questions = [
    {
        "question": "What is the recommended storage temperature for AKTA ready flow kits?",
        "relevant_pages": [1],
        "expected_doc_type": "Cover Letter"
    },

    {
        "question": "What is the lot number of the AKTA ready Low Flow Kit?",
        "relevant_pages": [2],
        "expected_doc_type": "Certificate of Quality"
    },

    {
        "question": "What material replaced PVC in Revision C?",
        "relevant_pages": [5],
        "expected_doc_type": "Packaging Specification"
    },

    {
        "question": "Do the AKTA ready flow kits contain materials of animal origin?",
        "relevant_pages": [6],
        "expected_doc_type": "BSE/TSE Declaration"
    },

    {
        "question": "When was the supplier's last on-site audit?",
        "relevant_pages": [8],
        "expected_doc_type": "Supplier Qualification"
    }
]

print("Evaluation test set created.")
print("Number of questions:", len(evaluation_questions))

Evaluation test set created.
Number of questions: 5


In [ ]:
# ============================================
# STEP 10B: Check Evaluation Document
# ============================================

print("Document store ready:", doc_store.is_ready)
print("Number of pages:", len(doc_store.pages))
print("Number of chunks:", len(doc_store.chunks))

Document store ready: False
Number of pages: 0
Number of chunks: 0


In [ ]:
# ============================================
# STEP 10C: Run Retrieval Evaluation
# ============================================
#
# WHAT WE'RE DOING:
# For each test question:
# 1. Ask FAISS for the top 4 chunks
# 2. Record their pages
# 3. Check whether each chunk is relevant
# 4. Record how long retrieval takes
#
# IMPORTANT:
# We are NOT calling RoBERTa or FLAN-T5 here.
# ============================================

import time
import pandas as pd

K = 4

evaluation_results = []

for test in evaluation_questions:

    question = test["question"]
    relevant_pages = test["relevant_pages"]

    # Start timer
    start_time = time.perf_counter()

    # Run ONLY retrieval
    retrieved, routed_type = retrieve_chunks(
        query=question,
        chunks=doc_store.chunks,
        index=doc_store.index,
        k=K,
        filter_doc_type=None,
        auto_route=False # Auto-route disabled because evaluation showed better retrieval performance: # Recall@4 improved 60% to 100% and MRR 0.60 to 0.90.
    )
    print("Question:", question)
    print("Retrieved chunks:", len(retrieved))

    # Stop timer
    retrieval_time = time.perf_counter() - start_time

    # Check each retrieved chunk
    for rank, (chunk, similarity) in enumerate(retrieved, start=1):

        is_relevant = (
            chunk.page_start in relevant_pages
            or chunk.page_end in relevant_pages
        )

        evaluation_results.append({
            "Question": question,
            "Rank": rank,
            "Retrieved Page": chunk.page_start,
            "Document Type": chunk.doc_type,
            "Similarity": round(float(similarity), 4),
            "Relevant": is_relevant,
            "Retrieval Time (sec)": round(retrieval_time, 4)
        })


results_df = pd.DataFrame(evaluation_results)

results_df

Question: What is the recommended storage temperature for AKTA ready flow kits?
Retrieved chunks: 0
Question: What is the lot number of the AKTA ready Low Flow Kit?
Retrieved chunks: 0
Question: What material replaced PVC in Revision C?
Retrieved chunks: 0
Question: Do the AKTA ready flow kits contain materials of animal origin?
Retrieved chunks: 0
Question: When was the supplier's last on-site audit?
Retrieved chunks: 0


""


In [ ]:
# ============================================
# STEP 10D: Calculate Retrieval Metrics
# ============================================

question_metrics = []

for test in evaluation_questions:

    question = test["question"]

    q_results = results_df[
        results_df["Question"] == question
    ].sort_values("Rank")

    relevant_flags = q_results["Relevant"].tolist()

    # -----------------------------
    # Recall@4 / Hit Rate
    # -----------------------------
    # Did at least one correct chunk appear
    # anywhere in the top 4?
    recall_at_k = 1 if any(relevant_flags) else 0

    # -----------------------------
    # Precision@4
    # -----------------------------
    # What fraction of the top 4 chunks
    # were actually relevant?
    precision_at_k = (
        sum(relevant_flags) / len(relevant_flags)
        if len(relevant_flags) > 0
        else 0
    )

    # -----------------------------
    # Reciprocal Rank
    # -----------------------------
    # Find the rank of the FIRST relevant chunk
    relevant_ranks = q_results.loc[
        q_results["Relevant"] == True,
        "Rank"
    ].tolist()

    if relevant_ranks:
        first_relevant_rank = min(relevant_ranks)
        reciprocal_rank = 1 / first_relevant_rank
    else:
        first_relevant_rank = None
        reciprocal_rank = 0

    retrieval_time = q_results[
        "Retrieval Time (sec)"
    ].iloc[0]

    question_metrics.append({
        "Question": question,
        "Found in Top 4": bool(recall_at_k),
        "First Relevant Rank": first_relevant_rank,
        "Precision@4": precision_at_k,
        "Reciprocal Rank": reciprocal_rank,
        "Retrieval Time (sec)": retrieval_time
    })


metrics_df = pd.DataFrame(question_metrics)

# Overall metrics
recall_at_4 = metrics_df["Found in Top 4"].mean()
precision_at_4 = metrics_df["Precision@4"].mean()
mrr = metrics_df["Reciprocal Rank"].mean()
avg_retrieval_time = metrics_df["Retrieval Time (sec)"].mean()

print("========== RETRIEVAL EVALUATION ==========")
print(f"Questions evaluated: {len(evaluation_questions)}")
print(f"Recall@4: {recall_at_4:.1%}")
print(f"Precision@4: {precision_at_4:.1%}")
print(f"MRR: {mrr:.3f}")
print(f"Average Retrieval Time: {avg_retrieval_time:.4f} sec")

metrics_df

KeyError: 'Question'

In [ ]:

# ============================================
# STEP 11: Gradio Callback Functions
# ============================================
#
# These functions connect the backend to the UI.
#
# They return only normal Python / Gradio values.
# The FAISS index stays inside doc_store.
# ============================================

def _document_type_dropdown(choices, value="All"):
    return gr.Dropdown(
        choices=choices,
        value=value,
        label="Document Type Filter",
        info="Limit retrieval to one detected document type."
    )


def process_documents_ui(files):
    success, stats = doc_store.process_files(files)

    if not success:
        error = stats.get("error", "Unknown processing error.")

        return (
            f"""
### Processing failed

**{error}**

Check the Colab output for the full error message.
""",
            [],
            _document_type_dropdown(["All"]),
            "Processing failed."
        )

    filenames = ", ".join(stats["filenames"])
    doc_types = ", ".join(stats["document_types"]) or "Other"

    info_md = f"""
### Successfully processed

**File(s):** {filenames}
**Pages:** {stats["total_pages"]}
**Documents found:** {stats["documents_found"]}
**Chunks created:** {stats["total_chunks"]}
**Document types:** {doc_types}
**OCR pages:** {stats["ocr_pages"]}
**Processing time:** {stats["processing_time"]:.1f} s
"""

    structure_rows = []

    for row in doc_store.get_document_structure():
        structure_rows.append([
            row["doc_type"],
            row["source_file"],
            row["pages"],
            row["chunks"]
        ])

    filter_choices = [
        "All"
    ] + stats["document_types"]

    return (
        info_md,
        structure_rows,
        _document_type_dropdown(filter_choices),
        (
            f"Ready — {stats['documents_found']} document(s), "
            f"{stats['total_chunks']} searchable chunk(s)."
        )
    )


def ask_document_ui(
    question,
    history,
    document_filter,
    auto_route,
    top_k,
    response_mode
):
    if history is None:
        history = []

    history = list(history)

    if not question or not question.strip():
        return (
            history,
            "",
            [],
            "No query yet.",
            history
        )

    result = doc_store.query(
        question=question.strip(),
        document_filter=document_filter,
        auto_route=auto_route,
        k=int(top_k),
        mode=response_mode
    )

    history.append({
        "role": "user",
        "content": question.strip()
    })

    # ========================================
    # Build answer for a standard chatbot
    # ========================================

    assistant_message = result["answer"]


    # ------------------------------
    # Add sources under the answer
    # ------------------------------

    if result.get("sources"):

        assistant_message += "\n\n**Sources:**\n"

        for source in result["sources"]:

            relevance = (
                source["similarity"] * 100
            )

            assistant_message += (

                f"- {source['doc_type']} "
                f"(Page {source['pages']}) "
                f"- Relevance: {relevance:.2f}%\n"
            )


    # ------------------------------
    # Add confidence
    # ------------------------------

    confidence = (
        result.get(
            "confidence",
            0.0
        )
        * 100
    )


    assistant_message += (

        f"\n*Confidence: "
        f"{confidence:.1f}%*"
    )


    # ------------------------------
    # Put everything in chatbot
    # ------------------------------

    history.append({

        "role":
            "assistant",

        "content":
            assistant_message
    })

    source_rows = []

    for source in result.get("sources", []):
        source_rows.append([
            source["source_id"],
            source["source_file"],
            source["doc_type"],
            source["pages"],
            source["similarity"],
            "Yes" if source["ocr"] else "No"
        ])

    confidence = result.get("confidence", 0.0)
    chunks_used = result.get("chunks_used", 0)
    route = result.get("route", "All documents")

    metrics_md = f"""
**Confidence:** {confidence:.0%} &nbsp;&nbsp; | &nbsp;&nbsp;
**Chunks used:** {chunks_used} &nbsp;&nbsp; | &nbsp;&nbsp;
**Route:** {route}
"""

    return (
        history,
        "",
        source_rows,
        metrics_md,
        history
    )


def export_chat_history(history):
    if not history:
        return None

    export_path = "/content/healthdoc_chat_history.json"

    with open(export_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "created_at": datetime.now().isoformat(),
                "application": "HealthDoc AI",
                "chat_history": history
            },
            f,
            indent=2,
            ensure_ascii=False
        )

    return export_path


def clear_application():
    doc_store.clear()

    return (
        None,
        "No documents processed yet.",
        [],
        _document_type_dropdown(["All"]),
        [],
        "",
        [],
        "No query yet.",
        "Upload PDFs, then click Process documents.",
        [],
        None
    )


print("Gradio callback functions ready.")


Gradio callback functions ready.


In [ ]:

# ============================================
# STEP 12: Clean Professional Gradio Interface
# ============================================
# This version intentionally uses STANDARD Gradio components and restrained CSS.
# No giant decorative upload/chat icons are added.
# The layout contains exactly the requested product functions:
#   • PDF upload
#   • Document Info
#   • Document Type Filter
#   • Auto-Route Queries
#   • Chunks to Retrieve
#   • Chat History
#   • Sources
#   • Confidence
#   • Export History

CUSTOM_CSS = """
.gradio-container {
    max-width: 1380px !important;
    margin: 0 auto !important;
    font-family: Inter, ui-sans-serif, system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif !important;
}
.header-card {
    border: 1px solid #e5e7eb;
    border-radius: 14px;
    padding: 18px 22px;
    margin: 8px 0 16px 0;
    background: white;
}
.section-card {
    border: 1px solid #e5e7eb !important;
    border-radius: 12px !important;
    padding: 14px !important;
    background: white !important;
}
.muted { color: #667085; font-size: 13px; }
"""

with gr.Blocks(
    title="HealthDoc AI",
    theme=gr.themes.Soft(primary_hue="teal", neutral_hue="slate"),
    css=CUSTOM_CSS
) as demo:

    chat_history_state = gr.State([])

    gr.HTML("""
    <div class="header-card">
      <div style="display:flex;align-items:center;justify-content:space-between;gap:16px;">
        <div>
          <div style="font-size:25px;font-weight:750;color:#162033;">HealthDoc AI</div>
          <div class="muted">Open-source healthcare document intelligence</div>
        </div>
        <div style="font-size:12px;font-weight:650;color:#087f73;background:#ecfdf8;
                    border:1px solid #ccefe7;border-radius:999px;padding:7px 12px;">
          RAG · Sources · Confidence
        </div>
      </div>
    </div>
    """)

    with gr.Row(equal_height=False):

        # ---------------- LEFT SIDE ----------------
        with gr.Column(scale=2, min_width=340):

            with gr.Group(elem_classes="section-card"):
                gr.Markdown("### PDF Document Viewer")
                gr.Markdown(
                    "Upload one or more digital or scanned PDF files. "
                    "OCR is used automatically when a page has little extractable text."
                )

                pdf_files = gr.File(
                    label="Upload PDF documents",
                    file_types=[".pdf"],
                    file_count="multiple",
                    type="filepath"
                )

                with gr.Row():
                    process_button = gr.Button("Process Documents", variant="primary")
                    clear_button = gr.Button("Clear All")

                processing_status = gr.Markdown(
                    "Upload PDFs, then click **Process Documents**."
                )

            with gr.Group(elem_classes="section-card"):
                gr.Markdown("### Document Info")
                document_info = gr.Markdown("No documents processed yet.")

                document_structure = gr.Dataframe(
                    headers=["Document Type", "Source File", "Pages", "Chunks"],
                    datatype=["str", "str", "str", "number"],
                    value=[],
                    interactive=False,
                    label="Detected Documents"
                )

            with gr.Group(elem_classes="section-card"):
                gr.Markdown("### Settings")

                doc_filter = gr.Dropdown(
                    choices=["All"],
                    value="All",
                    label="Document Type Filter",
                    info="Search all documents or only one detected document type."
                )

                auto_route = gr.Checkbox(
                    value=False,
                    label="Auto-Route Queries",
                    info="Automatically choose the most relevant document type."
                )

                top_k = gr.Slider(
                    1, 10, value=4, step=1,
                    label="Chunks to Retrieve",
                    info="Choose how many relevant chunks are sent to the answer model."
                )

                response_mode = gr.Dropdown(
                    choices=["Q&A Mode", "Summary Mode"],
                    value="Q&A Mode",
                    label="Response Mode"
                )

        # ---------------- RIGHT SIDE ----------------
        with gr.Column(scale=5, min_width=650):

            with gr.Group(elem_classes="section-card"):
                gr.Markdown("### Ask Questions")
                gr.Markdown(
                    "Answers are grounded in retrieved document chunks. "
                    "The conversation remains visible below."
                )

                chatbot = gr.Chatbot(
                    label="Chat History",
                    height=430,
                    type="messages",
                    show_label=True
                )

                question = gr.Textbox(
                    label="Question",
                    placeholder="Example: What packaging configuration changes were made?",
                    lines=2
                )

                ask_button = gr.Button("Ask Question", variant="primary")

                gr.Examples(
                    examples=[
                        ["What is the main purpose of this document?"],
                        ["What packaging configuration changes were made?"],
                        ["What quality or compliance requirements are listed?"],
                        ["What storage or handling conditions are specified?"]
                    ],
                    inputs=[question]
                )

            with gr.Group(elem_classes="section-card"):
                gr.Markdown("### Answer Evidence")

                answer_metrics = gr.Markdown(
                    "**Confidence:** —  |  **Chunks used:** —  |  **Route:** —"
                )

                source_table = gr.Dataframe(
                    headers=["Source", "File", "Document Type", "Page(s)", "Similarity", "OCR"],
                    datatype=["str", "str", "str", "str", "number", "str"],
                    value=[],
                    interactive=False,
                    label="Retrieved Sources"
                )

            with gr.Group(elem_classes="section-card"):
                gr.Markdown("### Export Chat History")
                export_button = gr.Button("Export History")
                export_file = gr.File(label="Download JSON", interactive=False)

    # ---------- EVENTS ----------
    process_button.click(
        fn=process_documents_ui,
        inputs=[pdf_files],
        outputs=[document_info, document_structure, doc_filter, processing_status]
    )

    ask_button.click(
        fn=ask_document_ui,
        inputs=[question, chatbot, doc_filter, auto_route, top_k, response_mode],
        outputs=[chatbot, question, source_table, answer_metrics, chat_history_state]
    )

    question.submit(
        fn=ask_document_ui,
        inputs=[question, chatbot, doc_filter, auto_route, top_k, response_mode],
        outputs=[chatbot, question, source_table, answer_metrics, chat_history_state]
    )

    export_button.click(
        fn=export_chat_history,
        inputs=[chat_history_state],
        outputs=[export_file]
    )

    clear_button.click(
        fn=clear_application,
        inputs=[],
        outputs=[
            pdf_files, document_info, document_structure, doc_filter,
            chatbot, question, source_table, answer_metrics,
            processing_status, chat_history_state, export_file
        ]
    )

print("Clean Gradio interface created.")


Clean Gradio interface created.


In [ ]:

# ============================================
# STEP 13: Launch the App
# ============================================
# Use ONE server at a time.
#
# First we launch normally inside Colab. This avoids depending on
# Gradio's public tunnel just to test file upload.
#
# If you later need a public URL for submission, stop this cell and
# change share=False to share=True.

demo.queue(default_concurrency_limit=2).launch(
    share=False,
    debug=True,
    show_error=True
)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

Processing: pharma-blob-sample.pdf



## Test checklist

1. Run Step 1.
2. **Restart the Colab session once** after installation.
3. Run Steps 2–13 in order.
4. In the app, upload a small PDF.
5. Click **Process Documents**.
6. Confirm **Document Info** fills in.
7. Test **Document Type Filter**, **Auto-Route Queries**, and **Chunks to Retrieve**.
8. Ask a question and confirm **Chat History**, **Confidence**, **Chunks used**, **Route**, and **Retrieved Sources** appear.
9. Export the chat history.

### Public link for submission
First confirm the app works with `share=False`. Then stop Step 13 and change only `share=False` to `share=True`.

If `share=True` says **Could not create share link**, that is the Gradio tunnel service failing; it is separate from the PDF-processing code.
